# TP2 - Roubos de veiculos no RJ (2025 x 2024)

Notebook de conferencia: CSV, JSON/YAML e tokens. Rode as celulas de cima para baixo, com o Jupyter aberto na pasta `codigo/`.

A nota tecnica e o documento de previsao de tokens sao colados por voce no Word. O nome do autor entra so no Word, nao nos prompts.


In [1]:
import json
from pathlib import Path

import pandas as pd
import tiktoken
import yaml

# Preencha com o que aparecer no ChatGPT no dia do uso
ferramenta = "ChatGPT (OpenAI)"
modelo = "GPT-6"
data_uso = "04/11/2026"

caminho_csv = "../data/BaseDPEvolucaoMensalCisp.csv"
pasta_saidas = Path("../saidas")
pasta_entregaveis = Path("../entregaveis")
caminho_prompts = "../prompts/roteiro_prompts_crispdm.md"

janela = 1050000
fonte_janela = "https://developers.openai.com/api/docs/models/gpt-6-astra"
data_consulta_janela = "04/11/2026"
palavras_previstas = 500

texto_nota = ""
pal_nota = None
tok_nota = None

enc = tiktoken.get_encoding("o200k_base")


def formatar(n):
    return f"{int(n):,}".replace(",", ".")


def variacao(antes, depois):
    return round((depois - antes) / antes * 100, 1)


print(ferramenta, "|", modelo, "|", data_uso)
print("CSV existe?", Path(caminho_csv).exists())


ChatGPT (OpenAI) | GPT-6 | 04/11/2026
CSV existe? True


## 1. Leitura do CSV

Separador ponto e virgula. Encoding latin-1 (nao UTF-8).


In [2]:
df = pd.read_csv(caminho_csv, sep=";", encoding="latin-1")
print("Linhas e colunas:", df.shape)

dados = df[df["ano"].isin([2024, 2025])].copy()
print("Linhas 2024 e 2025:", len(dados))
print("Meses 2024:", sorted(dados.loc[dados["ano"] == 2024, "mes"].unique()))
print("Meses 2025:", sorted(dados.loc[dados["ano"] == 2025, "mes"].unique()))
print(dados.groupby(["ano", "fase"]).size())

assert sorted(dados.loc[dados["ano"] == 2024, "mes"].unique()) == list(range(1, 13))
assert sorted(dados.loc[dados["ano"] == 2025, "mes"].unique()) == list(range(1, 13))
assert (dados["fase"] == 3).all()
print("Recorte ok: 12 meses em cada ano, todos na fase 3.")


Linhas e colunas: (38410, 65)
Linhas 2024 e 2025: 3288
Meses 2024: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Meses 2025: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
ano   fase
2024  3       1644
2025  3       1644
dtype: int64
Recorte ok: 12 meses em cada ano, todos na fase 3.


## 2. Grafia da regiao

Sem unificar, Grande Niteroi vira duas regioes e o quadro passa de 8 para 9 linhas.


In [3]:
print("Regioes antes:")
print(dados["regiao"].value_counts())

dados.loc[dados["regiao"].str.startswith("Grande Niter"), "regiao"] = "Grande Niteroi"

print("Regioes depois:")
print(dados["regiao"].value_counts())


Regioes antes:
regiao
Interior                         1608
Capital                           984
Baixada Fluminense                456
Grande Niterói                    150
Grande NiterÃÂÃÂÃÂÃÂ³i      90
Name: count, dtype: int64
Regioes depois:
regiao
Interior              1608
Capital                984
Baixada Fluminense     456
Grande Niteroi         240
Name: count, dtype: int64


## 3. Quadro por regiao e ano (8 registros)

Totais da rubrica: roubos 30930 / 25235 e recuperados 19034 / 17406.


In [4]:
colunas = ["roubo_veiculo", "recuperacao_veiculos", "furto_veiculos", "roubo_carga"]
quadro = dados.groupby(["regiao", "ano"], as_index=False)[colunas].sum()
print(quadro.to_string(index=False))

assert len(quadro) == 8

totais = dados.groupby("ano")[colunas].sum()
print("Estado:")
print(totais)

assert int(totais.loc[2024, "roubo_veiculo"]) == 30930
assert int(totais.loc[2025, "roubo_veiculo"]) == 25235
assert int(totais.loc[2024, "recuperacao_veiculos"]) == 19034
assert int(totais.loc[2025, "recuperacao_veiculos"]) == 17406
print("Quadro conferido.")


            regiao  ano  roubo_veiculo  recuperacao_veiculos  furto_veiculos  roubo_carga
Baixada Fluminense 2024          10921                  6138            4451         1334
Baixada Fluminense 2025           8719                  5030            4504         1286
           Capital 2024          17302                  9960            8872         1769
           Capital 2025          13713                  9148            8680         1496
    Grande Niteroi 2024           1939                  1264            1993          248
    Grande Niteroi 2025           2058                  1367            2191          243
          Interior 2024            768                  1672            2021           86
          Interior 2025            745                  1861            2115           88
Estado:
      roubo_veiculo  recuperacao_veiculos  furto_veiculos  roubo_carga
ano                                                                   
2024          30930                 1903

## 4. JSON e YAML (gabarito do pandas)

O JSON e o YAML do ChatGPT sao colados na secao 6.


In [5]:
pasta_saidas.mkdir(exist_ok=True)
registros = []
for _, linha in quadro.iterrows():
    registros.append({
        "regiao": str(linha["regiao"]),
        "ano": int(linha["ano"]),
        "roubo_veiculo": int(linha["roubo_veiculo"]),
        "recuperacao_veiculos": int(linha["recuperacao_veiculos"]),
        "furto_veiculos": int(linha["furto_veiculos"]),
        "roubo_carga": int(linha["roubo_carga"]),
    })

caminho_json = pasta_saidas / "quadro_regiao_ano.json"
caminho_yaml = pasta_saidas / "quadro_regiao_ano.yaml"
caminho_json.write_text(json.dumps(registros, ensure_ascii=False, indent=2), encoding="utf-8")
caminho_yaml.write_text(yaml.safe_dump(registros, allow_unicode=True, sort_keys=False), encoding="utf-8")

lido_json = json.loads(caminho_json.read_text(encoding="utf-8"))
lido_yaml = yaml.safe_load(caminho_yaml.read_text(encoding="utf-8"))
assert lido_json == registros
assert lido_yaml == registros
assert len(lido_json) == 8
print("JSON e YAML salvos com", len(lido_json), "registros.")


JSON e YAML salvos com 8 registros.


## 5. Variacoes, razao e Sao Goncalo


In [6]:
est = {}
for col in colunas:
    v24 = int(totais.loc[2024, col])
    v25 = int(totais.loc[2025, col])
    est[col] = {"2024": v24, "2025": v25, "delta": v25 - v24, "pct": variacao(v24, v25)}
    print(col, formatar(v24), "->", formatar(v25), est[col]["pct"], "%")

razao_2024 = round(est["recuperacao_veiculos"]["2024"] / est["roubo_veiculo"]["2024"] * 100, 1)
razao_2025 = round(est["recuperacao_veiculos"]["2025"] / est["roubo_veiculo"]["2025"] * 100, 1)
print("Razao recuperados/roubados:", razao_2024, "% /", razao_2025, "% (nao e taxa)")

print("Roubo de veiculo por regiao:")
reg_roubo = {}
ordem = ["Capital", "Baixada Fluminense", "Grande Niteroi", "Interior"]
for regiao in ordem:
    v24 = int(quadro[(quadro["regiao"] == regiao) & (quadro["ano"] == 2024)]["roubo_veiculo"].iloc[0])
    v25 = int(quadro[(quadro["regiao"] == regiao) & (quadro["ano"] == 2025)]["roubo_veiculo"].iloc[0])
    reg_roubo[regiao] = {"2024": v24, "2025": v25, "pct": variacao(v24, v25)}
    print(regiao, formatar(v24), "->", formatar(v25), reg_roubo[regiao]["pct"], "%")

gn = dados[dados["regiao"] == "Grande Niteroi"]
mun_gn = gn.groupby(["munic", "ano"])["roubo_veiculo"].sum().unstack()
print(mun_gn)
sg_nome = [m for m in mun_gn.index if "Gon" in str(m)][0]
sg24 = int(mun_gn.loc[sg_nome, 2024])
sg25 = int(mun_gn.loc[sg_nome, 2025])
print(sg_nome, sg24, "->", sg25, "de", reg_roubo["Grande Niteroi"]["2025"], "em 2025")


roubo_veiculo 30.930 -> 25.235 -18.4 %
recuperacao_veiculos 19.034 -> 17.406 -8.6 %
furto_veiculos 17.337 -> 17.490 0.9 %
roubo_carga 3.437 -> 3.113 -9.4 %
Razao recuperados/roubados: 61.5 % / 69.0 % (nao e taxa)
Roubo de veiculo por regiao:
Capital 17.302 -> 13.713 -20.7 %
Baixada Fluminense 10.921 -> 8.719 -20.2 %
Grande Niteroi 1.939 -> 2.058 6.1 %
Interior 768 -> 745 -3.0 %
ano          2024  2025
munic                  
Maricá         43    63
Niterói       338   279
São Gonçalo  1558  1716
São Gonçalo 1558 -> 1716 de 2058 em 2025


## 6. JSON e YAML do ChatGPT (FASE 3)

Cole os blocos devolvidos na FASE 3.


In [ ]:
json_modelo = """
"""

yaml_modelo = """
"""

if not json_modelo.strip() or not yaml_modelo.strip():
    print("Cole o JSON e o YAML da FASE 3 para validar.")
else:
    jm = json.loads(json_modelo)
    ym = yaml.safe_load(yaml_modelo)
    print("Registros JSON:", len(jm), "| YAML:", len(ym))
    assert len(jm) == 8
    assert jm == ym

    def soma(lista, ano, col):
        return sum(item[col] for item in lista if item["ano"] == ano)

    print("roubos 2024/2025:", soma(jm, 2024, "roubo_veiculo"), soma(jm, 2025, "roubo_veiculo"))
    print("recuperados 2024/2025:", soma(jm, 2024, "recuperacao_veiculos"), soma(jm, 2025, "recuperacao_veiculos"))
    assert soma(jm, 2024, "roubo_veiculo") == 30930
    assert soma(jm, 2025, "roubo_veiculo") == 25235
    assert soma(jm, 2024, "recuperacao_veiculos") == 19034
    assert soma(jm, 2025, "recuperacao_veiculos") == 17406
    print("JSON e YAML do modelo conferem com o CSV.")


## 7. Texto da nota (so para tokens)

Cole o texto da FASE 6 em `texto_nota` para medir palavras e tokens. A nota em si e colada no Word.


In [ ]:
texto_nota = """
"""

if not texto_nota.strip():
    pal_nota = None
    tok_nota = None
    print("Cole o texto da FASE 6 para medir palavras e tokens da nota.")
    print("A estimativa usa palavras_previstas =", palavras_previstas)
else:
    texto_nota = texto_nota.strip()
    pal_nota = len(texto_nota.split())
    tok_nota = len(enc.encode(texto_nota))
    print("Nota TP2 medida | palavras", pal_nota, "| tokens", tok_nota)


## 8. Previsao de tokens

Tokenizador tiktoken / o200k_base. A estimativa da nota usa `palavras_previstas`, definida no inicio. Copie os numeros impressos para o Word de previsao de tokens.


In [ ]:
nota_modelo = (
    "NOTA TECNICA INTERNA N. 14/2026 (documento ficticio, com numeros da base publica do ISP-RJ). "
    "Assunto: roubos e recuperacao de veiculos no estado do Rio de Janeiro, 1o semestre de 2026. "
    "1. Objetivo e metodo. Esta nota resume a evolucao dos roubos de veiculos, dos veiculos recuperados, "
    "dos furtos de veiculos e dos roubos de carga no estado do Rio de Janeiro, para apoiar a reuniao de "
    "planejamento do segundo semestre. A comparacao usa os meses de janeiro a junho de 2026 e os mesmos "
    "meses de 2025, de modo que os dois periodos tenham o mesmo tamanho e a mesma sazonalidade. Os dados "
    "vem da serie mensal por area de delegacia (CISP) do ISP-RJ. Os meses de julho a setembro de 2026 "
    "estao em outra fase de consolidacao na base (fase 2) e ficaram fora da comparacao. "
    "2. Resultados no estado. Os roubos de veiculos passaram de 11.736 para 13.430, um aumento de 1.694 "
    "casos (14,4%). Os veiculos recuperados passaram de 8.658 para 10.116, um aumento de 16,8%. Os roubos "
    "de carga subiram de 1.574 para 1.733 (10,1%). Os furtos de veiculos, ao contrario, cairam de 8.417 "
    "para 7.656 (queda de 9,0%). "
    "3. Resultados por regiao. A Capital concentra o maior numero de roubos de veiculos: 6.614 em 2025 e "
    "7.461 em 2026 (aumento de 12,8%). A Baixada Fluminense foi a unica regiao com queda, de 4.027 para "
    "3.931 (2,4%). O Interior passou de 333 para 462 (38,7%). A Grande Niteroi passou de 762 para 1.576 "
    "(106,8%); o municipio de Sao Goncalo responde por 1.396 dos 1.576 casos da regiao em 2026. "
    "4. Recuperacao de veiculos. A razao entre veiculos recuperados e veiculos roubados no estado foi de "
    "73,8% em 2025 e de 75,3% em 2026. Essa razao nao e uma taxa de recuperacao: os veiculos recuperados "
    "incluem tambem veiculos furtados e veiculos roubados antes do periodo ou em outra regiao. "
    "5. Limitacoes. A base traz registros de ocorrencias, e nao o total de crimes que de fato aconteceram. "
    "Nao ha informacao sobre frota nem sobre populacao, de modo que nao e possivel calcular taxas por "
    "veiculo ou por habitante. A base tambem nao informa as causas das variacoes, e esta nota nao as "
    "atribui a nenhuma politica, operacao ou mudanca no registro das ocorrencias. "
    "6. Encaminhamentos. Recomenda-se: (a) verificar com a area responsavel pelos registros a serie de "
    "Sao Goncalo antes de divulgar o resultado da Grande Niteroi; (b) repetir a analise com os meses de "
    "julho a setembro quando a base for consolidada; (c) produzir um boletim em linguagem acessivel ao "
    "publico geral."
)

pal_modelo = len(nota_modelo.split())
tok_modelo = len(enc.encode(nota_modelo))
razao_tp = round(tok_modelo / pal_modelo, 3)
est_nota = int(round(palavras_previstas * razao_tp))
print("Nota modelo | palavras", pal_modelo, "| tokens", tok_modelo, "| tokens/palavra", razao_tp)
print("Estimativa da nota TP2:", palavras_previstas, "x", razao_tp, "=", est_nota)
if pal_nota is None:
    print("Nota TP2 ainda nao colada na secao 7.")
else:
    print("Nota TP2 medida | palavras", pal_nota, "| tokens", tok_nota)

linhas_csv = Path(caminho_csv).read_text(encoding="latin-1").splitlines()
amostra = "\n".join(linhas_csv[:1001])
tok_amostra = len(enc.encode(amostra))
chars_amostra = len(amostra)
chars_por_token = round(chars_amostra / tok_amostra, 4)
texto_csv = Path(caminho_csv).read_text(encoding="latin-1")
est_csv = int(round(len(texto_csv) / chars_por_token))
tok_csv = len(enc.encode(texto_csv))
print("CSV amostra 1000 linhas | chars", chars_amostra, "| tokens", tok_amostra, "| chars/token", chars_por_token)
print("CSV inteiro estimado", est_csv, "| medido", tok_csv)


In [ ]:
roteiro = Path(caminho_prompts).read_text(encoding="utf-8")
blocos = []
partes = roteiro.split("```")
for i, parte in enumerate(partes):
    if i % 2 == 1:
        blocos.append(parte.strip())
print("Prompts encontrados:", len(blocos))

nomes = [
    "1. Entendimento do negocio",
    "2. Entendimento dos dados",
    "3. Preparacao dos dados",
    "3b. Correcao JSON/YAML",
    "4. Modelagem",
    "5. Avaliacao",
    "4. Retorno",
    "6. Implantacao",
]
saida_prevista = [900, 900, 1100, 900, 1100, 900, 700, 1400]

acum = 0
linhas_tab = []
for i, bloco in enumerate(blocos):
    entrada = len(enc.encode(bloco))
    saida = saida_prevista[i] if i < len(saida_prevista) else 900
    acum += entrada + saida
    nome = nomes[i] if i < len(nomes) else f"prompt {i+1}"
    linhas_tab.append({"fase": nome, "entrada": entrada, "saida_prevista": saida, "acumulado": acum})
    print(nome, "| entrada", entrada, "| saida prev.", saida, "| acum.", acum)

prev_total = acum
print("Acumulado previsto:", prev_total, "de", janela, round(prev_total / janela * 100, 2), "%")
print("Cabe na janela?", prev_total < janela)
print("Fonte:", fonte_janela, "| consulta:", data_consulta_janela)

chat_path = None
for nome_arq in ["chat_exportado.md", "chat_exportado.txt", "chat_exportado.html"]:
    candidato = pasta_entregaveis / nome_arq
    if candidato.exists():
        chat_path = candidato
        break

if chat_path is None:
    print("Chat ainda nao exportado.")
    real_total = None
    pct_diff = None
else:
    chat_txt = chat_path.read_text(encoding="utf-8", errors="ignore")
    real_total = len(enc.encode(chat_txt))
    pct_diff = round((real_total - prev_total) / prev_total * 100, 1)
    print("Tokens reais:", real_total, "| previsto:", prev_total, "| diff %:", pct_diff)

print("Copie estes numeros para entregaveis/Previsao_Tokens.docx.")
